# 文献利用与方法审查
只读核对当前数据库和已保存预测；不执行模型拟合或第三方代码。不同配置/目录/观察数量不相加；非空不等于科学准入。外部作者保存成绩不是本Notebook复现。

In [1]:
from pathlib import Path
import collections, hashlib, json, statistics, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'data/cycling_db.py').exists(): ROOT = ROOT.parent
assert (ROOT / 'data/cycling_db.py').exists()
sys.path.insert(0, str(ROOT))
from data.cycling_db import open_database
OUT = ROOT / '文献利用与外部方法审查_20261021'
read = lambda p: json.loads(p.read_text(encoding='utf-8'))
saved = read(OUT / 'local_audit.json')


In [2]:
with open_database() as db:
    for sql in ['select count(*) from papers', 'select count(*) from literature_registry', 'select count(*) from document_sources', 'select count(*) from model_endpoints']:
        print(sql, db.conn.execute(sql).fetchone()[0])
    counts = {}
    for name in ['all_review', 'review_3e_all', 'reported_3e', 'reported_2e']:
        rows = db.fetch(name).rows
        counts[name] = (len(rows), len({r['source_group'] for r in rows}))
    assert counts == {'all_review':(273,178), 'review_3e_all':(116,85), 'reported_3e':(31,23), 'reported_2e':(26,21)}
    print('Tests / source groups:', counts)


select count(*) from papers 126
select count(*) from literature_registry 226
select count(*) from document_sources 270
select count(*) from model_endpoints 191
Tests / source groups: {'all_review': (273, 178), 'review_3e_all': (116, 85), 'reported_3e': (31, 23), 'reported_2e': (26, 21)}


In [3]:
with open_database() as db:
    rows = db.fetch().rows
    ids = {r['endpoint_id'] for r in rows}
    pending = [r for r in db.fetch('review_3e_all').rows if r['endpoint_id'] not in ids]
    assert len(pending) == 85
    print('Pending with null cycle lower bound:', sum(r.get('cycle_voltage_lower_V') is None for r in pending))
    coverage = {f:sum(r.get(f) is not None for r in rows) for f in ['BET_m2_g','total_pore_volume_cm3_g','N_at_pct','O_at_pct','mass_loading_mg_cm2','temperature_C']}
    print('Training field coverage:', coverage)
    assert all(coverage[f] == saved['inventory']['training_profiles']['reported_3e']['nonnull_fields'][f] for f in coverage)
    assert coverage['total_pore_volume_cm3_g'] == 18
    weights = collections.defaultdict(float)
    for r in rows: weights[r['source_group']] += r['source_weight']
    assert len(weights) == 23 and all(abs(w-1)<1e-12 for w in weights.values())
    print('Source total weights:', sorted(set(weights.values())), 'each share:', 1/23)


Pending with null cycle lower bound: 72
Training field coverage: {'BET_m2_g': 31, 'total_pore_volume_cm3_g': 18, 'N_at_pct': 0, 'O_at_pct': 0, 'mass_loading_mg_cm2': 0, 'temperature_C': 0}
Source total weights: [1.0] each share: 0.043478260869565216


In [4]:
pred = [r for r in read(ROOT / '初步模型训练_20261021/predictions.json') if r['variant'] == 'MLP/no_current']
assert len(pred) == 31
g = collections.defaultdict(list)
for r in pred: g[r['source_group']].append(r)
mae_by_source = {s:statistics.fmean(abs(r['observed']-r['predicted']) for r in v) for s,v in g.items()}
mse_by_source = {s:statistics.fmean((r['observed']-r['predicted'])**2 for r in v) for s,v in g.items()}
source = 'doi:10.1039/d4ta02115k'
share = mse_by_source[source] / sum(mse_by_source.values())
old = saved['saved_prediction_error_concentration']['MLP/no_current']
assert abs(statistics.fmean(mae_by_source.values())-old['mae'])<1e-12
assert abs(share-next(r['sse_share'] for r in old['sources'] if r['source_group']==source))<1e-12
print('MAE:', statistics.fmean(mae_by_source.values()), 'OPB source MAE:', mae_by_source[source], 'OPB SSE share:', share)
for directory in ['初步模型训练_20261021','模型诊断与改进_20261021']:
    manifest = read(ROOT / directory / 'run_manifest.json')
    assert all(hashlib.sha256((ROOT / x['path']).read_bytes()).hexdigest()==x['sha256'] for x in manifest['outputs'])
    print(directory, 'unchanged payloads', len(manifest['outputs']))


MAE: 2.850987411835315 OPB source MAE: 10.642474986486452 OPB SSE share: 0.4561276269458593
初步模型训练_20261021 unchanged payloads 277
模型诊断与改进_20261021 unchanged payloads 281


In [5]:
methods = read(OUT / 'repository_method_review.json')
assert methods['third_party_code_executed'] is False and methods['new_model_fits'] == 0
for repo in methods['reviewed_revisions']:
    print(repo['repository'], repo['commit'], repo.get('saved_outputs_not_reproduced', repo.get('split', repo.get('review_scope'))))
own = read(OUT / 'own_source_unused_features.json')
assert hashlib.sha256((ROOT / own['path']).read_bytes()).hexdigest()==own['sha256']
print('Own Table1 Raman:', {r['sample']:r['ID_IG'] for r in own['rows']})
print('Scientific fields changed: 0; new model fits: 0')


Shan-Zhu/ML-Carbon_Supercapacitor 2e7ca3e8b3755819c038fde1402741ab708a719f random rows;90/10
Shan-Zhu/ML-SC_3DCN f3094e00269513e97d64d45b6fd8c8f9c53b0e6a random rows;80/20
khub-crystalparadigm/capacitance_pred ca7fcfdb0f432d30312ccb87175fe17798f53030 {'random_split_r2': 0.9297, 'doi_split_r2': 0.1497, 'external_r2': 0.2349, 'external_mae_F_g': 386.34, 'external_rows': 31}
bhavya8823/AI-Based-Supercapacitor-Capacitance-Prediction b5a87fae1d10b421ff559eddd484412a6693e5a6 Repository README only
Own Table1 Raman: {'OPC': 0.84, 'OPB': 0.87, 'OPNP': 0.93, 'OPBNP': 0.99}
Scientific fields changed: 0; new model fits: 0
